In [7]:
data_table_name = "all_data"
dataset_name = "news_headlines"
id_field = "id"

In [8]:
# ysz.kedro_datasets.ProdigyLabeledExamples

import psycopg

with psycopg.connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT
                replace(encode(content, 'escape'), '\\"', '\"')::json #>> '{{meta,{id_field} }}' 
                AS {id_field}
            FROM
                link
            LEFT JOIN dataset ON dataset.id = link.dataset_id
            LEFT JOIN example ON example.id = link.example_id
            WHERE
            dataset.name = '{dataset_name}';
            """)

        labelled_examples = cur.fetchall()

In [9]:
labelled_examples

[('3',), ('4',), ('1',), ('0',)]

In [16]:
import json


with psycopg.connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            with x as (
                        SELECT
                            cast(replace(encode(content, 'escape'), '\\"', '\"')::json #>> '{{meta,{id_field} }}' AS INTEGER)
                            AS {id_field}
                        FROM
                            link
                        LEFT JOIN dataset ON dataset.id = link.dataset_id
                        LEFT JOIN example ON example.id = link.example_id
                        WHERE
                        dataset.name = '{dataset_name}'
            )
            select 
            -- or jsonb_pretty instead of json_serialize
            json_serialize(
                json_build_object('text', text, -- or to_json(text::text) etc 
                'meta', json_build_object('{id_field}', {id_field}
                                        , 'source', all_data.source
                )) 
            )
            from {data_table_name}
            where 
            {id_field} not in (select * from x)
            and text is not null 
            limit 5
            ;
            """)

        examples = [json.loads(x[0]) for x in cur.fetchall()]

In [17]:
examples

[{'text': 'Uber’s Lesson: Silicon Valley’s Start-Up Machine Needs Fixing',
  'meta': {'id': 0, 'source': 'The New York Times'}},
 {'text': 'Pearl Automation, Founded by Apple Veterans, Shuts Down',
  'meta': {'id': 1, 'source': 'The New York Times'}},
 {'text': 'How Silicon Valley Pushed Coding Into American Classrooms',
  'meta': {'id': 2, 'source': 'The New York Times'}},
 {'text': 'Women in Tech Speak Frankly on Culture of Harassment',
  'meta': {'id': 3, 'source': 'The New York Times'}},
 {'text': 'Silicon Valley Investors Flexed Their Muscles in Uber Fight',
  'meta': {'id': 4, 'source': 'The New York Times'}}]